# 04 Results Analysis

Notebook oddaniowy:
1. Trening top-3 modeli.
2. Predykcja i ensemble na te?cie.
3. Ewaluacja i wykresy niepewno?ci.


In [ ]:
from pathlib import Path
import sys


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "src").exists() and (p / "data").exists() and (p / "notebooks").exists():
            return p
    raise RuntimeError("Nie moge znalezc katalogu projektu. Uruchom notebook w repo projektu.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.core.config import DataConfig, OptunaConfig, TrainingConfig
from src.pipelines.train_top3 import run_top3_training_and_ensemble
from src.pipelines.evaluate_models import evaluate_saved_predictions


In [ ]:
final_summary = run_top3_training_and_ensemble(
    data_config=DataConfig(),
    optuna_config=OptunaConfig(),
    training_config=TrainingConfig(epochs=50, early_stopping_patience=10, device="cpu"),
)
final_summary


In [ ]:
eval_report = evaluate_saved_predictions(PROJECT_ROOT)
print("Generated evaluation entries:", len(eval_report))
list(eval_report.keys())[:5]


In [ ]:
summary_path = PROJECT_ROOT / "reports" / "final_results_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
summary.keys()


In [ ]:
ens_path = PROJECT_ROOT / "reports" / "predictions" / "ensemble_predictions.csv"
if ens_path.exists():
    ens_df = pd.read_csv(ens_path)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(ens_df["entropy"], bins=50, ax=ax[0])
    ax[0].set_title("Entropy distribution")
    sns.histplot(ens_df["margin"], bins=50, ax=ax[1])
    ax[1].set_title("Margin distribution")
    plt.tight_layout()
    plt.show()
else:
    print("Brak pliku ensemble_predictions.csv")
